In [8]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"
pio.templates.default = "ggplot2"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

from glob import glob

csvs = ["mnist-vgg-b10.csv", "mnist-vgg-b11.csv", "mnist-vgg-b20.csv"]

dfs = [pd.read_csv(csv) for csv in csvs]

for i, df in enumerate(dfs):
    df.drop(columns=["Unnamed: 0"], inplace=True, errors='ignore')
    df["dataset"] = csvs[i].split("-")[0]
    df["model"] = csvs[i].split("-")[1]
    df["epoch"] = int(csvs[i].split("-b")[-1].split(".csv")[0])
    df["name"] = csvs[i].split(".csv")[0]
    
data = pd.concat(dfs, ignore_index=True)
data.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,Simplification Threshold,Number of Minima,dataset,model,epoch,name
0,0.000000,64,mnist,vgg,10,mnist-vgg-b10
1,0.000048,63,mnist,vgg,10,mnist-vgg-b10
2,0.000057,62,mnist,vgg,10,mnist-vgg-b10
3,0.000060,61,mnist,vgg,10,mnist-vgg-b10
4,0.000096,60,mnist,vgg,10,mnist-vgg-b10


In [9]:
data = data.sort_values(by=["epoch", "Number of Minima"], ascending=False)
fig = make_subplots(rows=1, cols=3, subplot_titles=["Epoch 10 (83.4%)", "Epoch 11 (88.2%)", "Epoch 20 (95.29%)"], shared_yaxes=True)

for i, (e, df) in enumerate(data.groupby(["epoch"])):
	df = data[data["epoch"] == e].sort_values(by="Number of Minima", ascending=False)
	name = df["name"].unique()[0]

	trace = go.Scatter(x = df["Simplification Threshold"], y = df["Number of Minima"], mode="lines", line=dict(shape="hv"))

	fig.add_trace(
		trace,
		row=1,
		col=i+1,
	)

fig.update_annotations(font_size=16)
fig.update_layout(margin=dict(l=0, r=0, t=30, b=0), width=600*2, height=150*2, font=dict(size=14), showlegend=False)
fig.update_xaxes(title_text="Simplification Threshold", title_standoff=18, automargin=True)
fig.update_yaxes(title_text="#Valleys", type="log", title_standoff=18, automargin=True)
fig.write_image(f"plots/mnist-vgg-b10,11,20.png", scale=8)
# fig.show()